# Mixed mechanics locking qualification

Candidate verification, not a pre-existing success report. Runs pinned code in an isolated environment, repeats the original 80.2% failure alongside MINI and retains negative evidence. The original 15% screen is unchanged. No patient data. Read `docs/MIXED_MINI_DIAGNOSIS.md` and `docs/MIXED_MINI_ACCEPTANCE.md`.


In [1]:
import sys, os, time, pathlib, subprocess, json, hashlib, zipfile
ROOT = pathlib.Path('/content/CardiMech_mixed_'+time.strftime('%Y%m%d_%H%M%S'))
ROOT.mkdir()
CODE_COMMIT = 'beaf842e5f6518e74b69b03c40ece7f9ca8e0f93'
stages = {}
def run(args, name, cwd=None, timeout=7200):
    print('START', name, flush=True)
    try:
        p = subprocess.run(args, cwd=cwd, env=ENV if 'ENV' in globals() else None,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, timeout=timeout)
    except subprocess.TimeoutExpired as exc:
        raw = exc.stdout or ''
        (ROOT/name).write_text(raw.decode(errors='replace') if isinstance(raw,bytes) else raw)
        stages[name] = {'passed':False,'timeout':timeout}
        raise
    (ROOT/name).write_text(p.stdout)
    stages[name] = {'passed':p.returncode==0,'returncode':p.returncode}
    print(name,p.returncode,p.stdout[-4000:],flush=True)
    p.check_returncode()
source_paths = ["src/cardimech/fem_mixed.py","src/cardimech/fem_backend.py","tests/test_fem_mixed.py","tests/test_fem_material_regime.py","scripts/qualify_material_regimes.py","scripts/diagnose_locking.py","scripts/verify_mixed_locking.py","docs/MIXED_MINI_ACCEPTANCE.md","docs/MIXED_MINI_DIAGNOSIS.md"]
failure = None
try:
    run([sys.executable,'-m','pip','install','virtualenv==21.14.12'],'bootstrap.log')
    run([sys.executable,'-m','virtualenv',str(ROOT/'venv')],'venv.log')
    PY = str(ROOT/'venv/bin/python')
    ENV = os.environ.copy()
    ENV['PATH'] = str(ROOT/'venv/bin')+os.pathsep+ENV['PATH']
    ENV['OPENBLAS_NUM_THREADS'] = '1'
    REPO = ROOT/'CardiMech'
    run(['git','clone','https://github.com/Virelion-Biotech/Virelion-CardiMech.git',str(REPO)],'clone.log')
    run(['git','checkout',CODE_COMMIT],'checkout.log',REPO)
    run([PY,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pydantic==2.14.1',
         'pytest==8.4.2','ruff==0.17.1','-e',str(REPO)+'[dev,validation]',
         'git+https://github.com/Virelion-Biotech/Virelion-CardiInfer.git@9aabf87bdbee4c70b98229baa3bbea3fb896b490'],'install.log')
    run([PY,'-m','ruff','check','src','tests','scripts'],'lint.log',REPO)
    run([PY,'-m','ruff','format','--check','src','tests','scripts'],'format.log',REPO)
    run([PY,'-m','pytest','-q','--cov=cardimech','--cov-fail-under=80',
         '--junitxml='+str(ROOT/'suite.xml')],'suite.log',REPO)
    run([PY,'scripts/diagnose_locking.py','--output',str(ROOT/'linear-diagnosis')],'linear.log',REPO)
    run([PY,'scripts/verify_mixed_locking.py','--output',str(ROOT/'campaign')],'campaign.log',REPO)
    run([PY,'-m','pip','freeze'],'environment.txt',REPO)
except Exception as exc:
    failure = {'type':type(exc).__name__,'message':str(exc)}
finally:
    qualification_path = ROOT/'campaign/qualification.json'
    qualification = json.loads(qualification_path.read_text()) if qualification_path.exists() else None
    identity = {p:hashlib.sha256((REPO/p).read_bytes()).hexdigest() for p in source_paths
                if 'REPO' in globals() and (REPO/p).exists()}
    manifest = {'code_commit':CODE_COMMIT,'stages':stages,'failure':failure,
                'source_sha256':identity,'qualification':qualification,
                'accuracy_qualified':bool(qualification and qualification['passed'] and not failure),
                'scope':'Synthetic numerical qualification only; no patient/whole-heart/clinical accuracy claim',
                'archive_contents':'All generated case meshes/fields, logs, JUnit, qualification and exact source; no runtime environment directories'}
    (ROOT/'MANIFEST.json').write_text(json.dumps(manifest,indent=2)+'\n')
    archive = ROOT/'mixed_locking_evidence.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in ROOT.rglob('*'):
            if p.is_file() and p != archive and not p.is_relative_to(ROOT/'venv') and not p.is_relative_to(ROOT/'CardiMech'):
                z.write(p,str(p.relative_to(ROOT)))
        for p in source_paths:
            if 'REPO' in globals() and (REPO/p).exists(): z.write(REPO/p,'source/'+p)
    raw = archive.read_bytes()
    receipt = {'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest()}
    (ROOT/'receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
    print('EVIDENCE',json.dumps({'root':str(ROOT),'receipt':receipt,'accuracy_qualified':manifest['accuracy_qualified'],'failure':failure}),flush=True)
if failure:
    raise RuntimeError('Qualification did not pass; inspect retained logs and archive: '+str(failure))
assert manifest['accuracy_qualified']
print('MIXED_QUALIFICATION_PASSED',flush=True)


START bootstrap.log
bootstrap.log 0 Collecting virtualenv==21.14.12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 19.1 MB/s eta 0:00:00

START venv.log
venv.log 0 created virtual environment CPython3.13.16.final.0-64-x86_64 in 946ms
  creator CPython3Posix(dest=/content/CardiMech_mixed_20261022_052604/venv, clear=False, no_vcs_ignore=False, venv_redirect=None, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.2
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator

START clone.log
clone.log 0 Cloning into '/content/CardiMech_mixed_20261022_052604/CardiMech'...

START checkout.log
checkout.log 0 Note: switching to 'beaf842e5f6518e74b69b03c40ece7f9ca8e0f93'.

You are in 'detached HEAD' state. You can look around, make experimental
chan

In [2]:
# Evidence export only; runs after the unedited pinned qualification cell.
import xml.etree.ElementTree as ET
export_script = """import pathlib,json,numpy as np,hashlib,sys
root=pathlib.Path(sys.argv[1]);identities={}
keys=['reference_nodes','tetrahedra','displacement','mixed_pressure_Pa','bubble_displacement_m','element_J']
for p in root.rglob('spatial-mechanics.json'):
    if p.is_relative_to(root/'CardiMech') or p.is_relative_to(root/'venv'): continue
    raw=p.read_bytes();d=json.loads(raw)
    np.savez_compressed(p.with_name('state.npz'),**{k:np.asarray(d[k]) for k in keys if k in d})
    mesh=p.with_name('mesh.json')
    identities[str(p.relative_to(root))]={'field_sha256':hashlib.sha256(raw).hexdigest(),'field_bytes':len(raw),'mesh_sha256':hashlib.sha256(mesh.read_bytes()).hexdigest()}
(root/'bulk-field-identities.json').write_text(json.dumps(identities,indent=2))
"""
(ROOT/'export_state.py').write_text(export_script)
run([PY,str(ROOT/'export_state.py'),str(ROOT)],'state-export.log',REPO)
suite_counts={'tests':0,'failures':0,'errors':0,'skipped':0}
for s in ET.parse(ROOT/'suite.xml').getroot().iter('testsuite'):
    for k in suite_counts: suite_counts[k]+=int(s.attrib.get(k,0))
compact_manifest={'replay':manifest,'suite_counts':suite_counts,'full_archive_receipt':receipt,
    'compact_archive_contents':'Raw coordinates/connectivity/nodal displacement, mixed pressure, bubbles and element J as NPZ; logs/JUnit/qualification/source hashes; omitted full JSON and mesh files are identified by SHA-256 and regenerate from pinned code'}
(ROOT/'COMPACT_MANIFEST.json').write_text(json.dumps(compact_manifest,indent=2)+'\n')
compact_archive=ROOT/'mixed_locking_compact_evidence.zip'
with zipfile.ZipFile(compact_archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob('*'):
        if p.is_file() and not p.is_relative_to(ROOT/'venv') and not p.is_relative_to(REPO) and p.name not in {'mesh.json','spatial-mechanics.json','MANIFEST.json'} and p.suffix in {'.json','.log','.xml','.txt','.npz'}:
            z.write(p,str(p.relative_to(ROOT)))
    for p in source_paths: z.write(REPO/p,'source/'+p)
raw=compact_archive.read_bytes()
compact_receipt={'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest(),'git_blob':hashlib.sha1(b'blob '+str(len(raw)).encode()+b'\0'+raw).hexdigest()}
(ROOT/'compact-receipt.json').write_text(json.dumps(compact_receipt,indent=2)+'\n')
print('SUMMARY',json.dumps({'suite':suite_counts,'qualification_passed':manifest['accuracy_qualified'],'acceptance':manifest.get('qualification',{}).get('acceptance') if manifest.get('qualification') else None,'compact_receipt':compact_receipt,'full_receipt':receipt}),flush=True)
if manifest.get('qualification'):
    for tag in ['baseline','mixed']:
        print(tag,[(r['lambda_Pa'],r['level'],r.get('relative_displacement_error'),r.get('error')) for r in manifest['qualification'][tag]])


START state-export.log
state-export.log 0 
SUMMARY {"suite": {"tests": 146, "failures": 0, "errors": 0, "skipped": 0}, "qualification_passed": true, "acceptance": {"all_equilibria_passed": true, "finest_all_regimes_within_original_15_percent": true, "near_incompressible_error_improved_at_least_fourfold": true, "refinement_errors_decrease": true, "finest_quadrature_change_below_one_percent": true, "low_load_scaled_displacement_change_below_one_percent": true, "higher_bulk_error_within_original_15_percent": true, "finite_load_quadrature_change_below_one_percent": true}, "compact_receipt": {"bytes": 2197378, "sha256": "c8e1de7580c046e68e8541040230cf84286c1db64aace5a3173481a79bf3cb36", "git_blob": "51ccc624afa638a414b392244fddcca29e1359f7"}, "full_receipt": {"bytes": 21729648, "sha256": "e7599570a02d059bb452165ef973a1351352e427be1e2543f6ddc45c2611adbc"}}
baseline [(1000.0, 1, 0.20122526031102034, None), (1000.0, 2, 0.062156832925310365, None), (1000.0, 3, 0.021246228216065956, None), (1000